# ¿Para que esta esta Notebook?
Me he descarga un dataset de Kaggle con ventas de cafeterias, el dataset es como si hubiesen comprimido una base de datos en una sola tabla

Esta notebook esta hecha para poder dividir bien los datos, crear tablas nuevas y columnas nuevas, para así crear una base de datos mas "real"

La tabla de sales se descargo desde https://www.kaggle.com/datasets/ahmedabbas757/coffee-sales


Primero voy a editar el dataset original y crearé querys para MYSqL para que se guarden solas en la carpeta "Queries"

Solo habria que ejecutar esta notebook entera para crear las queries, ir a workbench y abrirlas para ejecutarlas

AVISO: Todas las tablas llevaran como prefijo el nombre de la tabla (product_name) ya que es la nomenclatura que me enseñaron y me gustaria escalar el proyecto

In [1]:
import pandas as pd
import random

In [2]:
# Orden de creacion: Categories - Products - Stores - Employees - Sales
# hay que empezar a crear por las que no tienen relaciones

#            categories (tabla de dimensiones)
#                 |
# (dimensiones)   |
# products -------|
#     |
#     |
#     |
# sales (tabla de hechos)
#   |
#   |------- stores (dimensiones)
#   |
#   |------- employees (dimensiones)

create_DB_and_categories = """
DROP DATABASE IF EXISTS coffeShopDB;        -- por si la bd ya estaba creada  
CREATE DATABASE IF NOT EXISTS coffeShopDB;
USE coffeShopDB;

-- CATEGORIES almacena las categorias de productos con su id y nombre
CREATE TABLE IF NOT EXISTS categories (
    category_id INT AUTO_INCREMENT PRIMARY KEY,
    category_name varchar(100) NOT NULL
);
"""

create_products = """
/*
    PRODUCTS guarda todos los productos a la venta,
    se relaciona con el category_id para conseguir los nombres de las categorias
    en un futuro
*/
CREATE TABLE IF NOT EXISTS products (
    product_id INT AUTO_INCREMENT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    product_type VARCHAR(100) NOT NULL,
    unit_price DECIMAL(10,2) NOT NULL,
    category_id INT NOT NULL,
    
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
    ON DELETE NO ACTION ON UPDATE CASCADE
);
"""
create_stores = """
/*
    STORES contiene información básica de las tiendas,
    las columnas nuevas son: "capacity" y "status".
    No venian en el dataset y pensé que estaría mejor así
*/
CREATE TABLE IF NOT EXISTS stores (
    store_id INT AUTO_INCREMENT PRIMARY KEY,
    store_location VARCHAR(100) NOT NULL,
    store_capacity INT NOT NULL,
    store_status BOOLEAN NOT NULL
);
"""
create_employees = """
/*
    EMPLOYEES es una tabla creada desde 0, ningun dato venía en el dataset original
    Está relacionada con store_id (la tienda en la que esta cada empleado)
*/
CREATE TABLE IF NOT EXISTS employees (
    employee_id INT AUTO_INCREMENT PRIMARY KEY,
    employee_name VARCHAR(100) NOT NULL,
    employee_hiring_date DATE NOT NULL,
    employee_position VARCHAR(80) NOT NULL,
    employee_dni varchar(10) NOT NULL UNIQUE,
    store_id INT NOT NULL,
    
    FOREIGN KEY (store_id) REFERENCES stores(store_id)
    ON DELETE NO ACTION ON UPDATE CASCADE
);
"""
create_sales = """
/*
    SALES (anteriormente TRANSACTIONS) contiene todas las ventas de las cafeterias,
    está relacionada con store para ver que tienda es en la que se vende,
    product para ver el producto y employee para ver quien lo vende.
    En el dataset un empleado solo puede vender un producto x cantidad,
    obviamente esto no pasa en la vida real pero se me complicaría mucho editarlo
    
    En MySQL añadiré sale_total y juntaré las columnas de DATE y Time
*/
CREATE TABLE IF NOT EXISTS sales (
    sale_id INT AUTO_INCREMENT PRIMARY KEY,
    sale_date DATE NOT NULL,
    sale_quantity INT NOT NULL,
    sale_time TIME NOT NULL,
    store_id INT NOT NULL,
    product_id INT NOT NULL,
    employee_id INT NOT NULL,
    
    FOREIGN KEY (store_id) REFERENCES stores(store_id)
    ON DELETE NO ACTION ON UPDATE CASCADE,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
    ON DELETE NO ACTION ON UPDATE CASCADE,
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
    ON DELETE NO ACTION ON UPDATE CASCADE
);
"""
with open("../Queries/01_schema.sql", "w" ,encoding="utf-8") as f:
    f.write(create_DB_and_categories)
    f.write(create_products)
    f.write(create_stores)
    f.write(create_employees)
    f.write(create_sales)

### Con el esquema acabado ahora falta rellenar los datos con todo lo del dataset

In [3]:
df = pd.read_csv("../Data/Coffee_Shop_Sales.csv",sep=";")

In [4]:
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_type,product_name,category_name
0,1,01/01/2023,7:06:11,2,5,Lower Manhattan,32,3,Gourmet brewed coffee,Ethiopia Rg,Coffee
1,2,01/01/2023,7:08:56,2,5,Lower Manhattan,57,"3,1",Brewed Chai tea,Spicy Eye Opener Chai Lg,Tea
2,3,01/01/2023,7:14:04,2,5,Lower Manhattan,59,"4,5",Hot chocolate,Dark chocolate Lg,Drinking Chocolate
3,4,01/01/2023,7:20:24,1,5,Lower Manhattan,22,2,Drip coffee,Our Old Time Diner Blend Sm,Coffee
4,5,01/01/2023,7:22:41,2,5,Lower Manhattan,57,"3,1",Brewed Chai tea,Spicy Eye Opener Chai Lg,Tea


### -- STORES -- 

In [11]:
# Insercion de datos en la tabla STORES

# el unico dato que recojo del dataset es store_location y creo que lo voy a cambiar por ciudades de España
# EN MySQL: HACER QUERY PARA ABRIR LA TIENDA DE NOJA Y METER EMPLEADOS NUEVOS
print(df["store_location"].unique()) # ['Lower Manhattan', 'Hell's Kitchen', 'Astoria'] originales

df["store_location"] = df["store_location"].replace({
    "Lower Manhattan": "Estepona",
    "Hell's Kitchen": "Torremolinos",
    "Astoria": "Noja"
})

insert_stores = """
USE coffeShopDB;
-- el  IGNORE es por si se ejecuta dos veces
INSERT IGNORE INTO stores (store_id,store_location, store_capacity, store_status) VALUES
(5, 'Estepona', 50, TRUE),
(8, 'Torremolinos', 30, TRUE),
(3, 'Noja', 60, FALSE); -- La de Noja estará cerrada y se abrirá con UPDATE
"""

with open("../Queries/02_data.sql", "w" ,encoding="utf-8") as f:
    f.write(insert_stores)

print(df["store_location"].unique())

<StringArray>
['Estepona', 'Torremolinos', 'Noja']
Length: 3, dtype: str
<StringArray>
['Estepona', 'Torremolinos', 'Noja']
Length: 3, dtype: str


### -- CATEGORIES -- 

In [12]:
# Insercion de datos en la tabla CATEGORIES

# en este caso solo nos importa el nombre de cada categoría
print(df["category_name"].unique()) # [Coffee, Tea, Drinking Chocolate, Bakery, Flavours, Loose Tea, Coffee beans, Packaged Chocolate, Branded]

insert_categories = """
-- Todas las categorias que puede tener un producto
INSERT IGNORE INTO categories (category_name) VALUES
('Coffee'),
('Tea'),
('Drinking Chocolate'),
('Bakery'),
('Flavours'),
('Loose Tea'),
('Coffee beans'),
('Packaged Chocolate'),
('Branded');
"""

with open("../Queries/02_data.sql", "a", encoding="utf-8") as f:
    f.write(insert_categories)

<StringArray>
[            'Coffee',                'Tea', 'Drinking Chocolate',
             'Bakery',           'Flavours',          'Loose Tea',
       'Coffee beans', 'Packaged Chocolate',            'Branded']
Length: 9, dtype: str


### -- EMPLOYEES --

In [13]:
# Insercion de datos en la tabla EMPLOYEES

# como esta tabla se ha creado desde 0, no se usará nada del datset obviamente
# HACER UNA QUERY PARA CREAR UNA COLUMNA NUEVA QUE SEA PARA APELLIDOS
# AÑADIR TAMBIEN UN DROP DEL 999 Y METER GENTE NUEVA PARA LA TIENDA DE NOJA

insert_employees = """
INSERT  IGNORE INTO employees (employee_id,employee_name, employee_hiring_date, employee_position, employee_dni, store_id) VALUES
(1,'Valeria Sánchez', '2022-03-15', 'Cashier', '12345678A', 5),
(2,'Jorge Moreno', '2021-11-08', 'Supervisor', '23456789B', 5),
(3,'Dario Fernández', '2023-01-20', 'Barista', '34567890C', 5),
(4,'Ana Torres', '2021-06-10', 'Barista', '45678901D', 5),
(5,'Marina Jiménez', '2024-02-05', 'Barista', '56789012E', 5),
(6,'David Ruiz', '2022-09-12', 'Barista', '67890123F', 8),
(7,'Lara Lépez', '2021-07-01', 'Supervisor', '78901234G', 8),
(8,'Javier Martín', '2023-05-18', 'Cashier', '89012345I', 8),
(999,'NoName', '2000-01-01', 'Cashier', '00000000A', 3);    -- empleado fantasma para la tienda de Noja (se borrará cuando abra la tienda)
"""

with open("../Queries/02_data.sql", "a",encoding="utf-8") as f:
    f.write(insert_employees)

### -- PRODUCTS --

In [14]:
# hay que ordenar  el producto por el id que tiene, tipo, nombre y precio por unidad
# no puedo crear ids nuevos pq ya estan asociados a ventas
# he tenido que guardarlo en csv y todo para poder hacerlo manualmente
products = (
    df[["product_id", "product_name", "product_type", "unit_price"]]
    .drop_duplicates()
    .sort_values("product_id")
)

print(products)

      product_id               product_name       product_type unit_price
3451           1        Brazilian - Organic      Organic Beans         18
4494           2   Our Old Time Diner Blend  House blend Beans         18
3968           3             Espresso Roast     Espresso Beans      14,75
3554           4       Primo Espresso Roast     Espresso Beans      20,45
4328           5     Columbian Medium Roast      Gourmet Beans         15
...          ...                        ...                ...        ...
4033          83  I Need My Bean! Latte cup         Housewares         14
7711          83  I Need My Bean! Latte cup         Housewares         23
3352          84            Chocolate syrup      Regular syrup        0,8
14            87       Ouro Brasileiro shot   Barista Espresso          3
8242          87       Ouro Brasileiro shot   Barista Espresso        2,1

[98 rows x 4 columns]


In [15]:
    # product_id INT AUTO_INCREMENT PRIMARY KEY,
    # product_name VARCHAR(100) NOT NULL,
    # product_type VARCHAR(100) NOT NULL,
    # product_size INT NOT NULL, -- se rellena en python ya que es inventado (1, 2, 3)
    # unti_price DECIMAL(10,2) NOT NULL,
    # category_id INT NOT NULL,
    
    # 1	Coffee
    # 2	Tea
    # 3	Drinking Chocolate
    # 4	Bakery
    # 5	Flavours
    # 6	Loose Tea
    # 7	Coffee beans
    # 8	Packaged Chocolate
    # 9	Branded
    
category_map = {
    "Coffee": 1,
    "Tea": 2,
    "Drinking Chocolate": 3,
    "Bakery": 4,
    "Flavours": 5,
    "Loose Tea": 6,
    "Coffee beans": 7,
    "Packaged Chocolate": 8,
    "Branded": 9}

# copia para no tocar el df original
products = df[["product_id", "product_name", "product_type", "unit_price", "category_name"]].copy()

# cambiar las comas por puntos
products["unit_price"] = (products["unit_price"].astype(str).str.replace(",", ".", regex=False).astype(float))

# crear category_id
products["category_id"] = products["category_name"].map(category_map)

# Si hay productos repetidos con precios diferentes nos quedamos con el precio mas alto
products = (
    products
    .sort_values("unit_price", ascending=False)
    .drop_duplicates(subset="product_id")
    .sort_values("product_id")
)

insert_products = """
INSERT  IGNORE INTO products
(product_id, product_name, product_type, unit_price, category_id)
VALUES
"""
values = []

for row in products.itertuples(index=False):

    product_name = row.product_name.replace("'", "''")
    product_type = row.product_type.replace("'", "''")

    values.append(
        f"({row.product_id}, "
        f"'{product_name}', "
        f"'{product_type}', "
        f"{row.unit_price:.2f}, "
        f"{row.category_id})"
    )

insert_products += ",\n".join(values) + ";"

with open("../Queries/02_data.sql", "a", encoding="utf-8") as f:
    f.write("\n\n")
    f.write(insert_products)

### -- SALES --

In [16]:
# Insercion de la tabla principal SALES
# Esta no va a ser manual ya que hay mas de 10000 filas de datos

# La tienda 3 con 999 pq como sea null peta
# 999 es empleado fantasma
employees_by_store = {
    5: [1, 2, 3, 4, 5],
    8: [6, 7, 8],
    3: [999]
}

sales = df.copy()
# metemos a un empleado random por cada tienda dependiendo de la tienda en la que trabaja
sales["employee_id"] = sales["store_id"].apply(
    lambda x: random.choice(employees_by_store[x])
)

# cambiar las fechas para que no de problemas
sales["transaction_date"] = pd.to_datetime(
    sales["transaction_date"],
    dayfirst=True
).dt.strftime("%Y-%m-%d")

insert_sales = """
INSERT  IGNORE INTO sales
(sale_date, sale_quantity, sale_time, store_id, product_id, employee_id) VALUES
"""

values = []

for row in sales.itertuples(index=False):

    values.append(
        f"('{row.transaction_date}', "
        f"{row.transaction_qty}, "
        f"'{row.transaction_time}', "
        f"{row.store_id}, "
        f"{row.product_id}, "
        f"{row.employee_id})"
    )

insert_sales += ",\n".join(values) + ";"

with open("../Queries/02_data.sql", "a", encoding="utf-8") as f:
    f.write("\n\n")
    f.write(insert_sales)